# Supervised Fine-tuning Mistral 7B Instruct v1.0 for FinFact Dataset

In [1]:
import torch
import pandas as pd
from peft import PeftModel
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
torch_version = torch.__version__
if torch_version == "2.0.1+cu118":
    print(f"Torch version is satisfied: {torch.__version__}")
else:
    print("Torch version should be 2.0.1+cu118. Please ensure that before going further")

Torch version is satisfied: 2.0.1+cu118


In [ ]:
# model_name = "mistralai/Mistral-7B-v0.1"
# peft_name = "../trl/output/adapter-weights"
# new_model_name = "finfact-nli"

# id2label = {0: "true", 1: "false", 2: "neutral"}
# label2id = {"true": 0, "false": 1, "neutral": 2}

# tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, model_max_length=1536,
#                                           add_special_tokens=False,
#                                           trust_remote_code=True)
# tokenizer.pad_token = "[PAD]"
# tokenizer.padding_side = "right"
# tokenizer.pad_token_id = 0

# model = AutoModelForSequenceClassification.from_pretrained(
#     model_name,
#     id2label=id2label,
#     label2id=label2id,
#     num_labels=3,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
#     trust_remote_code=False,
# )

# model = PeftModel.from_pretrained(model, peft_name)
# model = model.merge_and_unload()

# model.save_pretrained(new_model_name, safe_serialization=True)
# tokenizer.save_pretrained(new_model_name)

# model.push_to_hub(new_model_name, use_temp_dir=False)
# tokenizer.push_to_hub(new_model_name, use_temp_dir=False)

In [3]:
test = pd.read_csv("data/test_skeptic_labelsdf.csv")
# test.dropna(inplace=True)

In [4]:
test.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,text,labels
0,7BObqdlgd5A,TCS Q2 Results 2023-24 Highlights | TCS Share ...,5paisa,TCS has just announced its Quarterly Results. ...,"Hi guys, Quarter 2 FY24 result season ki shirv...","""Hi guys, the Q2 FY24 result season has begun ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggIDQcICA...,https://www.youtube.com/watch?v=7BObqdlgd5A,The financial influencer claims that TCS's Q2 ...,The claims made by the influencer are true if ...,<|prompter|>You are a Financial Contrarian Wri...,0
1,R_OryHP3Fcg,Israel-Hamas Conflict's Impact on India #shorts,5paisa,Gain insight into how the Israel-Hamas Conflic...,बिलियन डौलर का ट्रेटिंग रेलेशन्चिप खत्रे में ह...,The trading relationship between Israel and Ha...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=R_OryHP3Fcg,The financial influencer claims that the confl...,The claims made by the influencer are plausibl...,<|prompter|>You are a Financial Contrarian Wri...,1
2,qX3J9Tq0XYU,YOUTUBE SE INCOME || MY FIRST INCOME,Amrev,YOUTUBE SE INCOME || MY FIRST INCOME\r\n\r\n...,so yeah parents do say a lot but it's okay wh...,"""So yeah, parents do say a lot, but it's okay....",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=qX3J9Tq0XYU,No claims made,No claims to analyse,<|prompter|>You are a Financial Contrarian Wri...,2
3,uQc6Tib329A,Growpital Review - 16% TAX FREE Return | Fixed...,Shrija Saha,Fixed Income - 16% Tax FREE Return - Growpita...,Fixed deposit with 16% returns वो भी tax-free ...,"""Fixed deposits with a 16% return are not tax-...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=uQc6Tib329A\n,"The financial influencer claims that Gropetal,...",The claim of a 16% tax-free return in one year...,<|prompter|>You are a Financial Contrarian Wri...,1
4,SQJxDy9F_7o,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,Amrev,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,सबसे पहले मेरा एक बहुत अफरेंट सबवाल है कि जो आ...,"First of all, my question is what suggestions ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAcHBw...,https://www.youtube.com/watch?v=SQJxDy9F_7o,The financial influencer suggests that young i...,The claim that businesses with less risk have ...,<|prompter|>You are a Financial Contrarian Wri...,0


In [5]:
model_name = "amazon/MistralLite"
peft_model_name = "../trl/output/classify/adapter-weights"


tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = "[PAD]"
tokenizer.padding_side = "right"
tokenizer.pad_token_id = 0

id2label = {0: "true", 1: "false", 2: "neutral"}
label2id = {"true": 0, "false": 1, "neutral": 2}

model = AutoModelForSequenceClassification.from_pretrained(model_name, torch_dtype=torch.float16, id2label=id2label, 
                                                           label2id=label2id, num_labels=3,
                                                           device_map="auto", local_files_only=True)
model.load_adapter(peft_model_name)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at amazon/MistralLite and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
preds = []

for i in range(len(test)):
    with torch.no_grad():
        text = tokenizer.encode_plus(test.iloc[i]["Justification"], test.iloc[i]["Summary_Claims"],
                                     max_length=512, truncation=True, return_tensors="pt")
        logits = model(**text).logits
        predicted_class_id = logits.argmax().item()
        pred_label = model.config.id2label[predicted_class_id]
        preds.append(pred_label)

In [8]:
def mapping(x):
    s = x["labels"]
    if s == 0:
        return "true"
    elif s == 1:
        return "false"
    elif s == 2:
        return "neutral"


test["labels"] = test.apply(mapping, axis=1)

In [10]:
def compute_metrics(pred, labels):
    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    return {"accuracy": accuracy}


actuals = test["labels"].tolist()

mistral_results = compute_metrics(actuals, preds)
mistral_accuracy = mistral_results["accuracy"]*100

print(f"Accuracy score from Mistral-7B --> {mistral_accuracy} %")

Accuracy score from Mistral-7B --> 61.904761904761905 %


In [17]:
test["predicted_label"] = preds

In [18]:
test.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,text,labels,predicted_label
0,7BObqdlgd5A,TCS Q2 Results 2023-24 Highlights | TCS Share ...,5paisa,TCS has just announced its Quarterly Results. ...,"Hi guys, Quarter 2 FY24 result season ki shirv...","""Hi guys, the Q2 FY24 result season has begun ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggIDQcICA...,https://www.youtube.com/watch?v=7BObqdlgd5A,The financial influencer claims that TCS's Q2 ...,The claims made by the influencer are true if ...,<|prompter|>You are a Financial Contrarian Wri...,true,true
1,R_OryHP3Fcg,Israel-Hamas Conflict's Impact on India #shorts,5paisa,Gain insight into how the Israel-Hamas Conflic...,बिलियन डौलर का ट्रेटिंग रेलेशन्चिप खत्रे में ह...,The trading relationship between Israel and Ha...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=R_OryHP3Fcg,The financial influencer claims that the confl...,The claims made by the influencer are plausibl...,<|prompter|>You are a Financial Contrarian Wri...,false,true
2,qX3J9Tq0XYU,YOUTUBE SE INCOME || MY FIRST INCOME,Amrev,YOUTUBE SE INCOME || MY FIRST INCOME\r\n\r\n...,so yeah parents do say a lot but it's okay wh...,"""So yeah, parents do say a lot, but it's okay....",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=qX3J9Tq0XYU,No claims made,No claims to analyse,<|prompter|>You are a Financial Contrarian Wri...,neutral,neutral
3,uQc6Tib329A,Growpital Review - 16% TAX FREE Return | Fixed...,Shrija Saha,Fixed Income - 16% Tax FREE Return - Growpita...,Fixed deposit with 16% returns वो भी tax-free ...,"""Fixed deposits with a 16% return are not tax-...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=uQc6Tib329A\n,"The financial influencer claims that Gropetal,...",The claim of a 16% tax-free return in one year...,<|prompter|>You are a Financial Contrarian Wri...,false,true
4,SQJxDy9F_7o,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,Amrev,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,सबसे पहले मेरा एक बहुत अफरेंट सबवाल है कि जो आ...,"First of all, my question is what suggestions ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAcHBw...,https://www.youtube.com/watch?v=SQJxDy9F_7o,The financial influencer suggests that young i...,The claim that businesses with less risk have ...,<|prompter|>You are a Financial Contrarian Wri...,true,neutral


In [19]:
test.to_csv("claims_classify_labels_205_records.csv", index=False)

In [11]:
label_names = ['false', 'neutral', 'true']
print(f"Classification Report: \n {classification_report(actuals, preds, target_names=label_names)}")

Classification Report: 
               precision    recall  f1-score   support

       false       0.60      0.50      0.55         6
     neutral       0.62      0.83      0.71         6
        true       0.62      0.56      0.59         9

    accuracy                           0.62        21
   macro avg       0.62      0.63      0.62        21
weighted avg       0.62      0.62      0.61        21



In [12]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, classification_report, f1_score


def calculate_metrics(actuals, preditctions):
    precision = precision_score(actuals, preditctions, average='macro')
    accuracy = accuracy_score(actuals, preditctions)
    f1_scoree = f1_score(actuals, preditctions, average='macro')
    conf_matrix = confusion_matrix(actuals, preditctions)
    recall_metric = recall_score(actuals, preditctions, pos_label="true", average="macro")
    cls_report = classification_report(actuals, preditctions, labels=['false', 'neutral', 'true'])
    return precision, accuracy, f1_scoree, conf_matrix, recall_metric, cls_report

a, b, c, d, e, f = calculate_metrics(actuals, preds)

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1521: UserWarning: Note that pos_label (set to 'true') is ignored when average != 'binary' (got 'macro'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


In [13]:
print(f)

              precision    recall  f1-score   support

       false       0.60      0.50      0.55         6
     neutral       0.62      0.83      0.71         6
        true       0.62      0.56      0.59         9

    accuracy                           0.62        21
   macro avg       0.62      0.63      0.62        21
weighted avg       0.62      0.62      0.61        21



In [14]:
from collections import Counter
Counter(preds)

Counter({'true': 8, 'neutral': 8, 'false': 5})

In [15]:
Counter(actuals)

Counter({'true': 9, 'false': 6, 'neutral': 6})